# **INITIAL ANALYSIS**

In [1]:
import requests
import pandas as pd
from io import StringIO
import time
import os
import geopandas as gpd
import warnings

# ***Check of DB***

In [2]:
df_raw = pd.read_csv("data/INGV/ingv_raw_earthquakes_1985_2025_complete.csv")

In [3]:
print(f"The uploaded dataframe has {len(df_raw):_} rows and {len(df_raw.columns):_} columns.")
memory = df_raw.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"It occupies {memory:_.2f} MB in RAM memory")
print(df_raw.columns)
display( pd.concat([df_raw.head(5),df_raw.tail(5)]) )

The uploaded dataframe has 216_097 rows and 14 columns.
It occupies 89.80 MB in RAM memory
Index(['#eventid', 'time', 'latitude', 'longitude', 'depth/km', 'author',
       'catalog', 'contributor', 'contributorid', 'magtype', 'magnitude',
       'magauthor', 'eventlocationname', 'eventtype'],
      dtype='object')


,#eventid,time,latitude,longitude,depth/km,author,catalog,contributor,contributorid,magtype,magnitude,magauthor,eventlocationname,eventtype
0,2289,1985-01-28T08:45:53.200000,42.5150,13.3130,10.0,BULLETIN-VAX,NaN,NaN,NaN,Md,2.5,--,1 km SE Capitignano (AQ),earthquake
1,2249,1985-01-27T03:24:43.880000,40.7590,15.2500,10.0,BULLETIN-VAX,NaN,NaN,NaN,Md,2.9,--,3 km NW Valva (SA),earthquake
2,2159,1985-01-25T23:33:12.210000,39.1350,16.0000,99.7,BULLETIN-VAX,NaN,NaN,NaN,Md,3.1,--,Costa Calabra nord-occidentale (Cosenza),earthquake
3,2149,1985-01-25T22:26:11.110000,40.5710,19.3100,10.0,BULLETIN-VAX,NaN,NaN,NaN,Md,2.8,--,Costa Albanese settentrionale (ALBANIA),earthquake
4,2089,1985-01-23T16:11:34.960000,38.1700,20.6190,10.0,BULLETIN-VAX,NaN,NaN,NaN,M,4.3,--,Costa Greca Ionica (GRECIA),earthquake
216092,44752202,2025-12-01T10:35:34.450000,45.8927,11.9652,7.8,SURVEY-INGV,NaN,NaN,NaN,ML,1.5,--,3 km W Valdobbiadene (TV),earthquake
216093,44751392,2025-12-01T07:12:13.000000,42.7787,12.7258,10.4,SURVEY-INGV,NaN,NaN,NaN,ML,1.9,--,5 km N Spoleto (PG),earthquake
216094,44750742,2025-12-01T03:10:19.730000,45.8738,7.0510,11.1,SURVEY-INGV,NaN,NaN,NaN,ML,1.6,--,11 km NE Courmayeur (AO),earthquake
216095,44750662,2025-12-01T02:25:48.990000,44.1677,9.9072,6.4,SURVEY-INGV,NaN,NaN,NaN,ML,1.7,--,1 km NW Santo Stefano di Magra (SP),earthquake
216096,44750592,2025-12-01T01:27:50.160000,44.5130,10.2288,25.0,SURVEY-INGV,NaN,NaN,NaN,ML,2.4,--,2 km E Tizzano Val Parma (PR),earthquake


In [4]:
df_raw['eventtype'].value_counts()

eventtype
earthquake                215291
quarry blast                 648
controlled explosion          87
experimental explosion        41
explosion                     18
landslide                      4
anthropogenic event            3
volcanic eruption              2
not reported                   1
accidental explosion           1
other event                    1
Name: count, dtype: int64

# ***OPTIONAL ANALYSIS FILTER ONLY ITALY EARTH DATA***

# ***OPTIONAL ANALISYS FILTER ONLY ITALY EARTH AND SEA DATA***

In [5]:
import pandas as pd
import geopandas as gpd
import warnings

warnings.filterwarnings('ignore')

print("Loading earthquake data...")
df = pd.read_csv("data/INGV/ingv_raw_earthquakes_1985_2025_complete.csv")

# 1. Create GeoDataFrame (EPSG:4326 is Lat/Lon in degrees)
gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326"
)

print("Fetching geographical boundaries...")
url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)
italy_land = world[world['ADMIN'] == "Italy"]

# --- THE MAGIC FOR THE SEA HAPPENS HERE ---

# 2. Project Italy to a Metric System (UTM Zone 32N - EPSG:32632 is perfect for Italy)
# This converts our map from degrees to meters so we can measure exact distances
italy_metric = italy_land.to_crs(epsg=32632)

# 3. Apply the Buffer to include the sea
# 12 Nautical Miles = approx 22,224 meters. 
# We'll use 30,000 meters (30 km) just to be safe and catch coastal fault lines
BUFFER_METERS = 30000 
print(f"Expanding borders by {BUFFER_METERS/1000} km to include territorial waters...")
italy_with_sea_metric = italy_metric.buffer(BUFFER_METERS)

# 4. Project back to standard GPS coordinates (Degrees) to match our earthquake points
italy_with_sea = italy_with_sea_metric.to_crs(epsg=4326)

# -------------------------------------------

print("Filtering points (checking Land + Sea)...")
# Check points against the new, expanded polygon
is_in_italy_and_sea = gdf.within(italy_with_sea.unary_union)

# 5. Filter the dataframe
df_italy_sea = df[is_in_italy_and_sea].copy()

print(f"\n--- Filtering Results ---")
print(f"Original events: {len(df)}")
print(f"Events in Italy (Land + 30km Sea): {len(df_italy_sea)}")
print(f"Events removed (far away): {len(df) - len(df_italy_sea)}")

# Save this definitively!
df_italy_sea.to_csv("data/INGV/ingv_italy_and_sea_1985_2025.csv", index=False)
print("\nFinal dataset saved successfully!")

Loading earthquake data...
Fetching geographical boundaries...
Expanding borders by 30.0 km to include territorial waters...
Filtering points (checking Land + Sea)...

--- Filtering Results ---
Original events: 216097
Events in Italy (Land + 30km Sea): 198909
Events removed (far away): 17188

Final dataset saved successfully!
